In [0]:
import pandas as pd
from pyspark.sql import functions as F
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np

df_spark = spark.table("cleaned_customer_data")

df = df_spark.toPandas()
#df.display()

In [0]:
#let's convert the binary fields to numeric '1/0'
df["repeat_customer_flag"] = (df["Repeat_Customer"] == "Yes").astype(int)
df["discount_flag"] = (df["Discount_Applied"] == "Yes").astype(int)

In [0]:
#let's extract the time features of purchase_date
df["Purchase_Date"] = pd.to_datetime(df["Purchase_Date"])
df["Purchase_DayOfWeek"] = df["Purchase_Date"].dt.dayofweek
df["Purchase_Month"] = df["Purchase_Date"].dt.month
df["Purchase_Hour"] = df["Purchase_Date"].dt.hour

In [0]:
# Drop rows with missing target
df.dropna(subset=["repeat_customer_flag"], inplace=True)

In [0]:
import pandas as pd

def aggregate_customer_data(df):

    # Ensure correct types
    df["Purchase_Amount"] = pd.to_numeric(df["Purchase_Amount"], errors="coerce")
    df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
    df["Purchase_Date"] = pd.to_datetime(df["Purchase_Date"], errors="coerce")
    df["Discount_Applied"] = df["Discount_Applied"].map({"Yes": 1, "No": 0})

    # Aggregate by Customer_ID
    customer_df = df.groupby("Customer_ID").agg({
        "Purchase_Amount": ["mean", "sum", "count"],
        "Discount_Applied": "mean",
        "Rating": "mean",
        "Purchase_Date": ["min", "max"],
        "Repeat_Customer": "max",  # Keep this as the label
        "Category": lambda x: x.mode()[0] if not x.mode().empty else None
    }).reset_index()

    # Flatten column names
    customer_df.columns = ["_".join(col).strip("_") for col in customer_df.columns.values]

    # Create custom features
    customer_df["Purchase_Span_Days"] = (customer_df["Purchase_Date_max"] - customer_df["Purchase_Date_min"]).dt.days
    customer_df["Repeat_Customer_Flag"] = customer_df["Repeat_Customer_max"].map({"Yes": 1, "No": 0})

    # Rename for clarity
    customer_df.rename(columns={
        "Purchase_Amount_mean": "Avg_Purchase_Amount",
        "Purchase_Amount_sum": "Total_Spend",
        "Purchase_Amount_count": "Total_Purchases",
        "Discount_Applied_mean": "Discount_Rate",
        "Rating_mean": "Avg_Rating",
        "Category_<lambda>": "Most_Common_Category"
    }, inplace=True)

    # Drop old columns
    customer_df.drop(columns=["Purchase_Date_min", "Purchase_Date_max", "Repeat_Customer_max"], inplace=True)

    return customer_df

customer_df = aggregate_customer_data(df)   

# Final check
print(customer_df.head())

In [0]:
X = customer_df[["Avg_Purchase_Amount",
"Total_Spend",
"Total_Purchases",
"Discount_Rate",
"Avg_Rating",
"Most_Common_Category",
"Purchase_Span_Days"]]

y = customer_df["Repeat_Customer_Flag"]

In [0]:
# Categorical encoding for the categorical features
categorical_cols = ["Most_Common_Category"]
numeric_cols = list(set(X.columns) - set(categorical_cols))

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

pipeline = Pipeline([
    ("preprocessor", preprocessor)
])



X_transformed = pipeline.fit_transform(X)

joblib.dump(pipeline,"pipeline.pkl")


In [0]:
#visualize our encoded categorical features
print(X_transformed)

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

X_train, X_test, y_train, y_test = train_test_split(
    X_transformed, y, test_size=0.2, random_state=35, stratify=y
)

In [0]:
#model = RandomForestClassifier(n_estimators=100, random_state=35)
#model.fit(X_train, y_train)

#y_pred = model.predict(X_test)

from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

In [0]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

#grid search 
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],
    "solver": ["lbfgs"],  # change to "liblinear" if you want to test l1
    "class_weight": [None, "balanced"],
    "max_iter": [1000]
}

# Create model
lr = LogisticRegression()

# Grid search with 5-fold cross-validation
grid = GridSearchCV(lr, param_grid, cv=5, scoring="recall", n_jobs=-1)
grid.fit(X_train, y_train)

# Best model and evaluation
best_lr = grid.best_estimator_
y_pred_best = best_lr.predict(X_test)

from sklearn.metrics import classification_report
print("Best Parameters:", grid.best_params_)
print("Classification Report:\n", classification_report(y_test, y_pred_best))

In [0]:
#Evaluation
print("Classification Report:")
print(classification_report(y_test, y_pred_lr))



In [0]:
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [0]:
#  Save model
joblib.dump(lr_model, "churn_model.pkl")
print("Model saved as churn_model.pkl")